In [2]:
# Push-up: Drive ZIP -> Extract -> FAST keypoints -> Rule labels -> CSV/PKL
# (학습은 포함하지 않음)
# ============================================

from google.colab import drive
drive.mount('/content/drive')

# 1) 경로 설정 (여기만 내 드라이브 경로에 맞게 수정)
ZIP_PATH   = "/content/drive/MyDrive/푸시업.zip"            # 드라이브에 올린 ZIP
IMG_ROOT   = "/content/drive/MyDrive/pushup/images"         # 압축 풀릴 폴더
OUTPUT_DIR = "/content/drive/MyDrive/pushup/outputs"        # 결과 저장 폴더
CSV_OUT    = f"{OUTPUT_DIR}/pushup_labels.csv"
PKL_OUT    = f"{OUTPUT_DIR}/pushup_dataset.pkl"

import os, sys, glob, math, pickle, zipfile, warnings
os.makedirs(IMG_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2) 안정 버전 설치 (충돌 방지: i3/CPU)
try:
    import cv2, mediapipe as mp, numpy as np, pandas as pd
except Exception:
    !{sys.executable} -q -m pip install --upgrade pip
    !{sys.executable} -q -m pip install \
        "numpy==1.26.4" \
        "protobuf==3.20.3" \
        "opencv-python-headless==4.9.0.80" \
        "mediapipe==0.10.14" \
        "pandas==2.2.2" \
        "tqdm==4.66.4" \
        "pyarrow==16.1.0"
    import cv2, mediapipe as mp, numpy as np, pandas as pd

from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
mp_pose = mp.solutions.pose

# 3) ZIP 추출 (이미 풀려있으면 건너뜀)
def safe_extract_zip(zip_path, out_dir):
    if not os.path.exists(zip_path):
        print(f"[WARN] ZIP 없음: {zip_path}")
        return
    if any(True for _ in glob.iglob(os.path.join(out_dir, "**", "*"), recursive=True)):
        print("[INFO] IMG_ROOT에 파일이 있어 추출 생략")
        return
    print(f"[INFO] 압축 해제: {zip_path} -> {out_dir}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)

safe_extract_zip(ZIP_PATH, IMG_ROOT)

# 4) 유틸
LANDMARK_NAMES = [
    'nose','left_eye_inner','left_eye','left_eye_outer','right_eye_inner','right_eye','right_eye_outer',
    'left_ear','right_ear','mouth_left','mouth_right',
    'left_shoulder','right_shoulder','left_elbow','right_elbow','left_wrist','right_wrist',
    'left_pinky','right_pinky','left_index','right_index','left_thumb','right_thumb',
    'left_hip','right_hip','left_knee','right_knee','left_ankle','right_ankle',
    'left_heel','right_heel','left_foot_index','right_foot_index'
]

def angle_3pt(a, b, c):
    try:
        ab = (a[0]-b[0], a[1]-b[1]); cb = (c[0]-b[0], c[1]-b[1])
        dot = ab[0]*cb[0] + ab[1]*cb[1]
        nab = math.hypot(*ab); ncb = math.hypot(*cb)
        cosang = max(-1., min(1., dot/((nab*ncb) + 1e-9)))
        return math.degrees(math.acos(cosang))
    except:
        return float("nan")

def dist(a,b): return float(math.hypot(a[0]-b[0], a[1]-b[1]))
def mid(p,q):  return ((p[0]+q[0])/2.0, (p[1]+q[1])/2.0)

# 속도 최적화: 긴 변 256 리사이즈
def fast_resize_keep_aspect(img, max_side=256):
    h, w = img.shape[:2]
    if max(h, w) <= max_side: return img
    if h >= w:
        new_h = max_side; new_w = int(w * (max_side / h))
    else:
        new_w = max_side; new_h = int(h * (max_side / w))
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

def extract_keypoints(img_bgr):
    img_small = fast_resize_keep_aspect(img_bgr, 256)
    with mp_pose.Pose(static_image_mode=True, model_complexity=1,
                      enable_segmentation=False, min_detection_confidence=0.5) as pose:
        res = pose.process(cv2.cvtColor(img_small, cv2.COLOR_BGR2RGB))
        if not res.pose_landmarks: return None, img_small.shape[:2]
        lm = res.pose_landmarks.landmark
        H, W = img_small.shape[:2]
        kps = np.array([[p.x*W, p.y*H, p.visibility] for p in lm], dtype=np.float32)
        return kps, (H, W)

# 5) 푸시업 라벨 규칙 (정자세=1, 오자세=0)
# - Torso straight: (어깨-엉덩이-발목) 각도 좌/우 165°~180°
# - Hip deviation: 엉덩이의 (어깨-발목 선) 수직편차 / 몸길이 ≤ 0.08
# - Hand stack: |x(손목)-x(어깨)| ≤ 0.2 * 어깨폭 (좌/우 중 1개 이상)
# - Elbow reasonable: 팔꿈치 각도 70°~170° 범위(좌/우 중 1개 이상)
#   (양측 모두 <60° 또는 >175°이면 불량)
# - Neck neutral(완화): 귀-어깨-엉덩이 각도 130°~200° (좌/우 중 1개라도)
def label_pushup(kps, hw):
    H, W = hw
    idx = {n:i for i,n in enumerate(LANDMARK_NAMES)}
    def P(name):
        j = idx[name]; return (float(kps[j,0]), float(kps[j,1]))

    LS, RS = P('left_shoulder'), P('right_shoulder')
    LH, RH = P('left_hip'), P('right_hip')
    LA, RA = P('left_ankle'), P('right_ankle')
    LE, RE = P('left_elbow'), P('right_elbow')
    LW, RW = P('left_wrist'), P('right_wrist')
    LEar, REar = P('left_ear'), P('right_ear')

    sh_center = mid(LS, RS); hip_center = mid(LH, RH); an_center = mid(LA, RA)
    shoulder_width = dist(LS, RS) + 1e-6
    body_len = dist(sh_center, an_center) + 1e-6

    # Torso straight
    ang_r = angle_3pt(RS, RH, RA)
    ang_l = angle_3pt(LS, LH, LA)
    torso_ok = (165 <= ang_r <= 180) and (165 <= ang_l <= 180)

    # Hip deviation
    def vertical_distance(p, a, b, eps=1e-6):
        import numpy as np
        a = np.array(a); b = np.array(b); p = np.array(p)
        ab = b - a
        if np.linalg.norm(ab) < eps: return float(np.linalg.norm(p - a))
        return float(abs(np.cross(ab, a - p)) / (np.linalg.norm(ab)))
    hip_dev = vertical_distance(hip_center, sh_center, an_center) / body_len
    hip_ok = (hip_dev <= 0.08)

    # Hand stack
    dx_l = abs(LW[0] - LS[0]) / shoulder_width
    dx_r = abs(RW[0] - RS[0]) / shoulder_width
    hand_ok = (dx_l <= 0.20) or (dx_r <= 0.20)

    # Elbow angle
    ang_el_l = angle_3pt(LW, LE, LS)
    ang_el_r = angle_3pt(RW, RE, RS)
    elbow_reasonable = (70 <= ang_el_l <= 170) or (70 <= ang_el_r <= 170)
    elbow_bad_constant = (ang_el_l < 60 and ang_el_r < 60) or (ang_el_l > 175 and ang_el_r > 175)

    # Neck neutral (완화)
    neck_l = angle_3pt(LEar, LS, LH)
    neck_r = angle_3pt(REar, RS, RH)
    neck_ok = (130 <= neck_l <= 200) or (130 <= neck_r <= 200)

    # 최종 판정: 핵심 4개 중 3개 이상 + 명백한 팔꿈치 불량 금지
    good = sum([torso_ok, hip_ok, hand_ok, elbow_reasonable]) >= 3 and not elbow_bad_constant
    label = 1 if good else 0

    flags = {
        "torso_ok": torso_ok, "hip_ok": hip_ok, "hand_ok": hand_ok,
        "elbow_reasonable": elbow_reasonable, "neck_ok": neck_ok,
        "ang_torso_r": ang_r, "ang_torso_l": ang_l,
        "ang_elbow_l": ang_el_l, "ang_elbow_r": ang_el_r,
        "hip_dev_ratio": hip_dev, "dx_l_sw": dx_l, "dx_r_sw": dx_r
    }
    return label, flags

# 6) 이미지 나열
def image_paths_from(root):
    exts = ("*.jpg","*.jpeg","*.png","*.bmp","*.webp")
    files = []
    for e in exts:
        files.extend(glob.glob(os.path.join(root, "**", e), recursive=True))
    return sorted(list(set(files)))

all_imgs = image_paths_from(IMG_ROOT)
print(f"[INFO] 이미지 개수: {len(all_imgs)}")

# 7) 처리 루프 -> CSV/PKL 저장
import pandas as pd
rows_feat, rows_label = [], []

for p in tqdm(all_imgs, desc="Push-up"):
    img = cv2.imread(p)
    if img is None: continue
    kps, hw = extract_keypoints(img)
    if kps is None: continue

    y, f = label_pushup(kps, hw)

    feat = {
        "path": p, "label": int(y),
        "ang_torso_r": f["ang_torso_r"], "ang_torso_l": f["ang_torso_l"],
        "ang_elbow_l": f["ang_elbow_l"], "ang_elbow_r": f["ang_elbow_r"],
        "hip_dev_ratio": f["hip_dev_ratio"], "dx_l_sw": f["dx_l_sw"], "dx_r_sw": f["dx_r_sw"],
        "torso_ok": int(f["torso_ok"]), "hip_ok": int(f["hip_ok"]),
        "hand_ok": int(f["hand_ok"]), "elbow_reasonable": int(f["elbow_reasonable"]),
        "neck_ok": int(f["neck_ok"])
    }
    rows_feat.append(feat)
    rows_label.append({"path": p, "label": int(y)})

df = pd.DataFrame(rows_feat)
df.to_csv(CSV_OUT, index=False)

import pickle
X_cols = ["ang_torso_r","ang_torso_l","ang_elbow_l","ang_elbow_r","hip_dev_ratio","dx_l_sw","dx_r_sw"]
X = df[X_cols].fillna(0.0).to_numpy(dtype=np.float32)
y = df["label"].fillna(0).to_numpy(dtype=np.int64)
with open(PKL_OUT, "wb") as f:
    pickle.dump({"X": X, "y": y, "feature_cols": X_cols, "csv": CSV_OUT}, f)

print(f"[DONE] 저장 완료\n- CSV: {CSV_OUT}\n- PKL: {PKL_OUT}\n총 샘플: {len(df)}")


Mounted at /content/drive
[INFO] 압축 해제: /content/drive/MyDrive/푸시업.zip -> /content/drive/MyDrive/pushup/images
[INFO] 이미지 개수: 7168


Push-up:   0%|          | 0/7168 [00:00<?, ?it/s]

[DONE] 저장 완료
- CSV: /content/drive/MyDrive/pushup/outputs/pushup_labels.csv
- PKL: /content/drive/MyDrive/pushup/outputs/pushup_dataset.pkl
총 샘플: 7030


In [7]:
from google.colab import drive
import os

MOUNT_PT = "/content/drive2"   # 비어있는 경로 사용
os.makedirs(MOUNT_PT, exist_ok=True)
drive.mount(MOUNT_PT)          # force_remount 불필요 (새 폴더니까 비어있음)

print("Mounted at:", MOUNT_PT)


Mounted at /content/drive2
Mounted at: /content/drive2


In [8]:
# 라벨링 산출물이 있는 폴더 (MyDrive 기준)
BASE = f"{MOUNT_PT}/MyDrive/pushup/outputs"
DATA_PKL = f"{BASE}/pushup_dataset.pkl"
DATA_CSV = f"{BASE}/pushup_labels.csv"

import os
print("PKL exists:", os.path.exists(DATA_PKL), DATA_PKL)
print("CSV exists:", os.path.exists(DATA_CSV), DATA_CSV)


PKL exists: True /content/drive2/MyDrive/pushup/outputs/pushup_dataset.pkl
CSV exists: True /content/drive2/MyDrive/pushup/outputs/pushup_labels.csv


In [9]:
# == Push-up 모델 학습: PKL 우선, 없으면 CSV 사용. 정/오자세 개수 출력 ==
import json, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

# 1) 데이터 로드
if os.path.exists(DATA_PKL):
    with open(DATA_PKL, "rb") as f: b = pickle.load(f)
    X = np.asarray(b["X"], dtype=np.float32)
    y = np.asarray(b["y"], dtype=np.int64)
    feature_cols = b.get("feature_cols")
    print(f"[LOAD] PKL -> X:{X.shape}, y:{y.shape}")
elif os.path.exists(DATA_CSV):
    df = pd.read_csv(DATA_CSV)
    default_cols = ["ang_torso_r","ang_torso_l","ang_elbow_l","ang_elbow_r","hip_dev_ratio","dx_l_sw","dx_r_sw"]
    feature_cols = [c for c in default_cols if c in df.columns]
    X = df[feature_cols].fillna(0).to_numpy(np.float32)
    y = df["label"].fillna(0).astype("int64").to_numpy()
    print(f"[LOAD] CSV -> X:{X.shape}, y:{y.shape}, cols={feature_cols}")
else:
    raise FileNotFoundError("라벨링 파일을 찾을 수 없습니다. 위 exists 출력 확인!")

# 2) 정/오자세 개수
def cnt(lbl):
    c = Counter(lbl); return {"good_1": int(c.get(1,0)), "bad_0": int(c.get(0,0)), "total": int(len(lbl))}
counts = {"all": cnt(y)}
print(f"[COUNT ALL] 정자세(1):{counts['all']['good_1']}  오자세(0):{counts['all']['bad_0']}  전체:{counts['all']['total']}")

# 3) 분할
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
counts["train"] = cnt(y_tr); counts["val"] = cnt(y_va)
print("[COUNT TRAIN]", counts["train"])
print("[COUNT VAL]  ", counts["val"])

# 4) 모델 2개
pipe_lr = Pipeline([("scaler", StandardScaler()),
                    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])
rf = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=3,
                            class_weight="balanced", n_jobs=-1, random_state=42)
pipe_lr.fit(X_tr, y_tr); rf.fit(X_tr, y_tr)

def eval_m(m, Xv, yv, name):
    prob = m.predict_proba(Xv)[:,1] if hasattr(m,"predict_proba") else m.decision_function(Xv)
    pred = (prob >= 0.5).astype(int)
    return {"name":name,
            "acc":float(accuracy_score(yv,pred)),
            "f1":float(f1_score(yv,pred,zero_division=0)),
            "precision":float(precision_score(yv,pred,zero_division=0)),
            "recall":float(recall_score(yv,pred,zero_division=0)),
            "roc_auc":float(roc_auc_score(yv,prob)),
            "cm":confusion_matrix(yv,pred).tolist()}, prob

m_lr, prob_lr = eval_m(pipe_lr, X_va, y_va, "LogReg")
m_rf, prob_rf = eval_m(rf,      X_va, y_va, "RandomForest")
best_name, best_model, best_metrics, best_prob = ("LogReg", pipe_lr, m_lr, prob_lr)
if m_rf["roc_auc"] > m_lr["roc_auc"]:
    best_name, best_model, best_metrics, best_prob = ("RandomForest", rf, m_rf, prob_rf)
print("[RESULT] LogReg:", m_lr)
print("[RESULT] RandomForest:", m_rf)
print("[SELECT] Best:", best_name)

# 5) 베스트 임계값(F1 최대)
best_thr, best_f1 = 0.5, -1.0
for t in np.linspace(0.1,0.9,33):
    f1 = f1_score(y_va, (best_prob>=t).astype(int), zero_division=0)
    if f1>best_f1: best_f1, best_thr = float(f1), float(t)

# 6) 산출물 저장(프론트/깃허브 공유)
OUT = BASE
with open(f"{OUT}/model.pkl","wb") as f:
    pickle.dump({"model":best_model,"feature_cols":feature_cols,"threshold":best_thr,
                 "meta":{"type":"pushup_binary","selected_model":best_name}}, f)
with open(f"{OUT}/feature_cols.json","w",encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)
with open(f"{OUT}/metrics.json","w",encoding="utf-8") as f:
    json.dump({"LogReg":m_lr,"RandomForest":m_rf,"selected":best_name,
               "best_threshold":best_thr,"best_val_f1":best_f1}, f, ensure_ascii=False, indent=2)
with open(f"{OUT}/counts.json","w",encoding="utf-8") as f:
    json.dump(counts, f, ensure_ascii=False, indent=2)

pd.DataFrame({"y_true":y_va,"prob_good":best_prob,
              "pred@0.50":(best_prob>=0.50).astype(int),
              f"pred@{best_thr:.2f}":(best_prob>=best_thr).astype(int)}
             ).to_csv(f"{OUT}/pred_val.csv", index=False)

print(f"[DONE] saved -> {OUT}")


[LOAD] PKL -> X:(7030, 7), y:(7030,)
[COUNT ALL] 정자세(1):2213  오자세(0):4817  전체:7030
[COUNT TRAIN] {'good_1': 1770, 'bad_0': 3854, 'total': 5624}
[COUNT VAL]   {'good_1': 443, 'bad_0': 963, 'total': 1406}
[RESULT] LogReg: {'name': 'LogReg', 'acc': 0.9082503556187767, 'f1': 0.8654848800834203, 'precision': 0.8042635658914729, 'recall': 0.9367945823927766, 'roc_auc': 0.9723189149783524, 'cm': [[862, 101], [28, 415]]}
[RESULT] RandomForest: {'name': 'RandomForest', 'acc': 0.9893314366998578, 'f1': 0.9833147942157954, 'precision': 0.9692982456140351, 'recall': 0.9977426636568849, 'roc_auc': 0.9989967394030599, 'cm': [[949, 14], [1, 442]]}
[SELECT] Best: RandomForest
[DONE] saved -> /content/drive2/MyDrive/pushup/outputs
